# Notebook 03d — Modelos No Lineales (Verdes + Azules)

**Target**: `log1p(Tumor_Telomere fusion rate)`  
**Features verdes**: firstResponse, treatment_agrupado, Mesenchimal_origin  
**Features azules añadidas**: cancer_type, biopsySite, hasSystemicPreTreatment, hasRadiotherapyPreTreatment, treatmentGiven, treatment1  
**Modelos**: Random Forest → XGBoost → LightGBM

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import os
os.chdir(r'C:\Users\pablo\OneDrive\Pictures\Documents\Universidad\TFG')

print('Librerías cargadas OK')
print('Directorio:', os.getcwd())

## 1. Carga y preparación de datos

In [ ]:
df = pd.read_csv('datos_limpios_modelo_a.csv')

df_verde = pd.read_excel('revision_color_code.xlsx', sheet_name='Sheet1',
                         usecols=['Patient_ID_IF_mod', 'firstResponse', 'treatment'])
df = df.merge(df_verde, on='Patient_ID_IF_mod', how='left')

df_azul = pd.read_excel('revision_color_code.xlsx', sheet_name='Sheet1',
                        usecols=['Patient_ID_IF_mod', 'biopsySite', 'hasSystemicPreTreatment',
                                 'hasRadiotherapyPreTreatment', 'treatmentGiven', 'treatment1'])
df = df.merge(df_azul, on='Patient_ID_IF_mod', how='left')

df_mes = pd.read_excel('Tumor_mesenchimal.xlsx')
df_mes.columns = ['cancer_type_code', 'Mesenchimal_origin']
df = df.merge(df_mes, on='cancer_type_code', how='left')

# Target
TARGET = 'Tumor_Telomere fusion rate (Gt-end/coverage*purity)'
y = np.log1p(df[TARGET])

# Preparación features
df['firstResponse'] = df['firstResponse'].fillna('Sin_dato')
df['Mesenchimal_origin'] = df['Mesenchimal_origin'].fillna('Sin_dato')
top15 = df['treatment'].value_counts().head(15).index.tolist()
df['treatment_agrupado'] = df['treatment'].apply(
    lambda x: x if x in top15 else ('Sin_dato' if pd.isna(x) else 'Otro')
)
df['hasSystemicPreTreatment'] = df['hasSystemicPreTreatment'].fillna('Sin_dato')
df['hasRadiotherapyPreTreatment'] = df['hasRadiotherapyPreTreatment'].fillna('Sin_dato')
df['treatmentGiven'] = df['treatmentGiven'].fillna('Sin_dato')
top10_biopsy = df['biopsySite'].value_counts().head(10).index.tolist()
df['biopsySite_agrupado'] = df['biopsySite'].apply(
    lambda x: x if x in top10_biopsy else ('Sin_dato' if pd.isna(x) else 'Otro')
)
top10_t1 = df['treatment1'].value_counts().head(10).index.tolist()
df['treatment1_agrupado'] = df['treatment1'].apply(
    lambda x: x if x in top10_t1 else ('Sin_dato' if pd.isna(x) else 'Otro')
)

cols_features = ['firstResponse', 'treatment_agrupado', 'Mesenchimal_origin',
                 'cancer_type', 'biopsySite_agrupado', 'hasSystemicPreTreatment',
                 'hasRadiotherapyPreTreatment', 'treatmentGiven', 'treatment1_agrupado']

X = pd.get_dummies(df[cols_features], drop_first=True)

# Limpiar nombres de columnas para LightGBM (no acepta caracteres especiales)
import re
X.columns = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in X.columns]

print(f'Shape X: {X.shape} | Shape y: {y.shape}')

## 2. Split train / test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 3. Función de evaluación

In [ ]:
def evaluar_modelo(nombre, modelo, X_tr, X_te, y_tr, y_te):
    modelo.fit(X_tr, y_tr)
    y_pred_log  = modelo.predict(X_te)
    # Deshacer log1p → métricas en espacio original
    y_pred_orig = np.expm1(y_pred_log)
    y_te_orig   = np.expm1(np.array(y_te))

    r2   = r2_score(y_te_orig, y_pred_orig)
    mae  = mean_absolute_error(y_te_orig, y_pred_orig)
    rmse = np.sqrt(mean_squared_error(y_te_orig, y_pred_orig))
    cv   = cross_val_score(modelo, X_tr, y_tr, cv=5, scoring='r2').mean()

    sep = '=' * 50
    print(sep)
    print(f'  {nombre}')
    print(sep)
    print(f'  R²  (test, espacio original):  {r2:.4f}')
    print(f'  R²  (CV-5, espacio log):       {cv:.4f}')
    print(f'  MAE  (espacio original):        {mae:.4f}')
    print(f'  RMSE (espacio original):        {rmse:.4f}')

    if hasattr(modelo, 'feature_importances_'):
        imp = pd.Series(modelo.feature_importances_, index=X_tr.columns)
        print('  Top 10 features más importantes:')
        print(imp.sort_values(ascending=False).head(10).to_string())

    return {'nombre': nombre, 'R2_test': r2, 'R2_cv': cv,
            'MAE': mae, 'RMSE': rmse, 'y_pred': y_pred_orig}

## 4. Random Forest

In [ ]:
res_rf = evaluar_modelo('Random Forest',
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    X_train, X_test, y_train, y_test)

## 5. XGBoost

In [ ]:
res_xgb = evaluar_modelo('XGBoost',
    XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbosity=0),
    X_train, X_test, y_train, y_test)

## 6. LightGBM

In [ ]:
res_lgbm = evaluar_modelo('LightGBM',
    LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1),
    X_train, X_test, y_train, y_test)

## 8. Optimización de Hiperparámetros de Hojas (Optuna)

Siguiendo la sugerencia del tribunal, se optimizan los **hiperparámetros de las hojas** de cada árbol mediante búsqueda bayesiana con Optuna (50 trials, CV-5).

| Modelo | Parámetros optimizados |
|--------|------------------------|
| Random Forest | `min_samples_leaf`, `max_leaf_nodes`, `min_weight_fraction_leaf` |
| XGBoost | `min_child_weight` |
| LightGBM | `min_child_samples` |

In [ ]:
import optuna
from sklearn.metrics import make_scorer
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Scorer en espacio original (deshace log1p antes de calcular R2)
def r2_espacio_original(y_true, y_pred):
    return r2_score(np.expm1(y_true), np.expm1(y_pred))

scorer_original = make_scorer(r2_espacio_original)

# -- Random Forest: hiperparametros de hojas --------------------------
def objective_rf(trial):
    model = RandomForestRegressor(
        n_estimators=100, random_state=42, n_jobs=-1,
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 100),
        max_leaf_nodes=trial.suggest_categorical("max_leaf_nodes", [None, 50, 100, 200, 500]),
        min_weight_fraction_leaf=trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.05),
    )
    return cross_val_score(model, X_train, y_train, cv=5, scoring=scorer_original).mean()

study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(objective_rf, n_trials=50)
print(f"RF   -> mejores parametros de hojas : {study_rf.best_params}")
print(f"RF   -> R2CV (espacio original)     : {study_rf.best_value:.4f}")

# -- XGBoost: hiperparametros de hojas --------------------------------
def objective_xgb(trial):
    model = XGBRegressor(
        n_estimators=100, random_state=42, n_jobs=-1, verbosity=0,
        min_child_weight=trial.suggest_int("min_child_weight", 1, 50),
    )
    return cross_val_score(model, X_train, y_train, cv=5, scoring=scorer_original).mean()

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=50)
print(f"XGB  -> mejor parametro de hoja     : {study_xgb.best_params}")
print(f"XGB  -> R2CV (espacio original)     : {study_xgb.best_value:.4f}")

# -- LightGBM: hiperparametros de hojas -------------------------------
def objective_lgbm(trial):
    model = LGBMRegressor(
        n_estimators=100, random_state=42, n_jobs=-1, verbose=-1,
        min_child_samples=trial.suggest_int("min_child_samples", 5, 200),
    )
    return cross_val_score(model, X_train, y_train, cv=5, scoring=scorer_original).mean()

study_lgbm = optuna.create_study(direction="maximize")
study_lgbm.optimize(objective_lgbm, n_trials=50)
print(f"LGBM -> mejor parametro de hoja     : {study_lgbm.best_params}")
print(f"LGBM -> R2CV (espacio original)     : {study_lgbm.best_value:.4f}")

# -- Reentrenar con mejores parametros --------------------------------
res_rf_tuned = evaluar_modelo(
    "RF (tuned hojas)",
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1, **study_rf.best_params),
    X_train, X_test, y_train, y_test
)
res_xgb_tuned = evaluar_modelo(
    "XGBoost (tuned hoja)",
    XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbosity=0, **study_xgb.best_params),
    X_train, X_test, y_train, y_test
)
res_lgbm_tuned = evaluar_modelo(
    "LightGBM (tuned hoja)",
    LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1, **study_lgbm.best_params),
    X_train, X_test, y_train, y_test
)

# -- Comparativa default vs tuned -------------------------------------
print()
print('=== COMPARATIVA: DEFAULT vs TUNED (hiperparametros de hojas) ===')
comp = pd.DataFrame([
    {"Modelo": "Random Forest",
     "R2_test default": res_rf["R2_test"],   "R2_test tuned": res_rf_tuned["R2_test"],
     "R2_CV default":   res_rf["R2_cv"],     "R2_CV tuned":   res_rf_tuned["R2_cv"]},
    {"Modelo": "XGBoost",
     "R2_test default": res_xgb["R2_test"],  "R2_test tuned": res_xgb_tuned["R2_test"],
     "R2_CV default":   res_xgb["R2_cv"],    "R2_CV tuned":   res_xgb_tuned["R2_cv"]},
    {"Modelo": "LightGBM",
     "R2_test default": res_lgbm["R2_test"], "R2_test tuned": res_lgbm_tuned["R2_test"],
     "R2_CV default":   res_lgbm["R2_cv"],   "R2_CV tuned":   res_lgbm_tuned["R2_cv"]},
]).set_index("Modelo")
print(comp.round(4).to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, res in zip(axes, [res_rf, res_xgb, res_lgbm]):
    y_pred = np.array(res['y_pred'])
    y_real = np.expm1(np.array(y_test))   # deshacer log1p
    ax.scatter(y_real, y_pred, alpha=0.3, s=10, color='darkorange')
    lim = max(y_real.max(), y_pred.max())
    ax.plot([0, lim], [0, lim], 'r--', linewidth=1)
    ax.set_xlabel('Real (espacio original)')
    ax.set_ylabel('Predicho (espacio original)')
    ax.set_title(f"{res['nombre']}\nR²={res['R2_test']:.4f}")

plt.tight_layout()
plt.savefig('notebooks/img/img_nolineales_azules.png', dpi=150, bbox_inches='tight')
plt.close()
print('Guardado: notebooks/img/img_nolineales_azules.png')

In [ ]:
resultados = pd.DataFrame([res_rf, res_xgb, res_lgbm]).set_index('nombre')
print('=== COMPARATIVA — NO LINEALES VERDES + AZULES ===')
print(resultados.drop('y_pred', axis=1).round(4).to_string())
print()
print(f'Mejor modelo: {resultados["R2_test"].idxmax()}')
print()
print('--- Comparativa con solo verdes (03b) ---')
print('Random Forest: R2_test=0.0062')
print('XGBoost:       R2_test=0.0103')
print('LightGBM:      R2_test=0.0229')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, res in zip(axes, [res_rf, res_xgb, res_lgbm]):
    y_pred = np.array(res['y_pred'])
    y_real = np.array(y_test)
    ax.scatter(y_real, y_pred, alpha=0.3, s=10, color='darkorange')
    lim = max(y_real.max(), y_pred.max())
    ax.plot([0, lim], [0, lim], 'r--', linewidth=1)
    ax.set_xlabel('Real (log1p)')
    ax.set_ylabel('Predicho (log1p)')
    ax.set_title(f"{res['nombre']}\nR²={res['R2_test']:.4f}")
plt.tight_layout()
plt.savefig('notebooks/img/img_nolineales_azules.png', dpi=150, bbox_inches='tight')
plt.close()
print('Guardado: notebooks/img/img_nolineales_azules.png')